In [ ]:
import time
import numpy as np
import torch
import hockey.hockey_env as h_env
import matplotlib.pyplot as plt
from memory import Memory, ExperienceMemory, PrioritizedMemory
from sac import SACAgent

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
env = h_env.HockeyEnv()
player2 = h_env.BasicOpponent(weak=False)

ac_space = env.action_space
o_space = env.observation_space
print(ac_space)
print(o_space)
print(list(zip(env.observation_space.low, env.observation_space.high)))

In [ ]:
max_episodes=int(1e5)
max_steps=500

buffer = PrioritizedMemory()
agent = SACAgent(env.observation_space.shape[0], env.action_space.shape[0]//2, noise_seq_len=int(1e5), actor_nvp=False, device = device)

In [ ]:
ob,_info = env.reset()
print(ob)
agent.actor(torch.FloatTensor(ob).unsqueeze(0).to(device))

In [ ]:
stats = []
losses = []

In [ ]:
update_every_steps = 50
step_counter = 0

for i in range(max_episodes):
    # print("Starting a new episode")    
    total_reward = []
    ob, _info = env.reset(True)
    obs_agent2 = env.obs_agent_two()
    done = False
    for t in range(max_steps):
        step_counter += 1
        with torch.no_grad():
            a1, _ = agent.act(ob)
        a1 = a1.cpu().numpy()[0]
        a2 = player2.act(obs_agent2)

        (ob_new, reward, done, trunc, _info) = env.step(np.hstack([a1,a2]))
        buffer.add((ob, a1, reward, ob_new, float(done)))
        total_reward.append(reward)
        ob=ob_new        
        obs_agent2 = env.obs_agent_two()
        if step_counter % update_every_steps == 0:
            agent.update(buffer)
        if done: 
            break
    stats.append([i,total_reward,t+1])
    
    if ((i-1)%20==0):
        print("{}: Reward: {}".format(i, np.sum(total_reward)))

In [ ]:
def running_mean(x, N):
    cumsum = np.cumsum(np.insert(x, 0, 0)) 
    return (cumsum[N:] - cumsum[:-N]) / float(N)

r = [np.sum(x[1]) for x in stats]
print(np.mean(r))
plt.plot(r)
plt.plot(np.arange(20, len(r)+1), running_mean(r, 20))
plt.show()

In [ ]:
env = h_env.HockeyEnv()

In [ ]:
player2 = h_env.BasicOpponent(weak=False)

In [ ]:
obs_buffer = []
reward_buffer=[]
obs, info = env.reset(True)
obs_agent2 = env.obs_agent_two()
for _ in range(251):
    env.render()
    with torch.no_grad():
        a1, _ = agent.act(ob)
        a1 = a1.cpu().numpy()[0]
    a2 = player2.act(obs_agent2)

    obs, r, d, t, info = env.step(np.hstack([a1,a2]))    
    obs_buffer.append(obs)
    reward_buffer.append(r)
    obs_agent2 = env.obs_agent_two()
    if d or t: 
        break
obs_buffer = np.asarray(obs_buffer)
reward_buffer = np.asarray(reward_buffer)

In [ ]:
env = h_env.HockeyEnv(1)

In [ ]:
# player1 = h_env.BasicOpponent()
player2 = h_env.HumanOpponent(env=env, player=2)

In [ ]:
obs, info = env.reset()
env.render()
time.sleep(1)
obs_agent2 = env.obs_agent_two()
# for _ in range(10000):
while True:
    time.sleep(0.2)
    env.render()
    with torch.no_grad():
        a1, _ = agent.act(obs)
        a1 = a1.cpu().numpy()[0]
    # a1 = player1.act(obs)
    a2 = player2.act(obs_agent2)
    obs, r, d, _, info = env.step(np.hstack([a1,a2]))    
    obs_agent2 = env.obs_agent_two()
    if d: break

In [ ]:
env.close()